# Cancer Stem Cell (CSC) Identification — Full Pipeline
**Dataset:** GSE176078 — Wu et al. 2021 Breast Cancer scRNA-seq (100,064 cells)

## Two-Stage Approach
| Stage | Method | Output |
|---|---|---|
| **Stage 1** | Standard scRNA-seq clustering + differential expression | CSC marker list (DE method) |
| **Stage 2** | Geneformer transformer fine-tuning + attention weights | CSC gene program (attention method) |
| **G5** | Comparison | Shared markers + novel Stage-2-only discoveries |

**Runtime:** ~2–3 hours on Colab T4 GPU

---
### Before running
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Optionally mount Google Drive (Cell 2) to save results permanently
3. Run all cells in order with **Runtime → Run all**

In [ ]:
# ── Cell 1: GPU check ─────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU')

In [ ]:
# ── Cell 2: (Optional) Mount Google Drive to persist results ──
USE_DRIVE = False   # Set True to save checkpoints + results to Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/CSC_Pipeline'
else:
    BASE = '/content/CSC_Pipeline'

import os
for d in ['data/raw', 'data/processed', 'data/geneformer/input',
          'results/figures', 'results/tables', 'results/geneformer_finetuned']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

print(f'Working directory: {BASE}')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────
# Takes ~3 minutes on first run
!pip install -q \
    'scanpy>=1.10' \
    'anndata>=0.10' \
    'harmonypy>=0.0.9' \
    'leidenalg>=0.10' \
    'python-igraph>=0.11' \
    'umap-learn>=0.5.6' \
    'decoupler>=1.8' \
    'scikit-misc>=0.3.0' \
    'mygene' \
    'huggingface_hub' \
    'datasets' \
    'transformers==4.46' \
    'accelerate' \
    'peft' \
    seaborn
print('Core packages installed.')

In [ ]:
# ── Cell 4: Install Geneformer ────────────────────────────────
import os
GENEFORMER_DIR = f'{BASE}/geneformer_repo'

if not os.path.exists(GENEFORMER_DIR):
    from huggingface_hub import snapshot_download
    print('Downloading Geneformer (~500 MB)...')
    snapshot_download(
        repo_id='ctheodoris/Geneformer',
        local_dir=GENEFORMER_DIR,
        ignore_patterns=['*.tar.gz'],
    )
    !pip install -q -e {GENEFORMER_DIR}
    print('Geneformer installed.')
else:
    !pip install -q -e {GENEFORMER_DIR}
    print('Geneformer already present, reinstalled.')

In [ ]:
# ── Cell 5: Download Wu et al. 2021 BRCA dataset from GEO ─────
# GSE176078 — 100,064 cells, 29,733 genes
# File: ~1 GB compressed, ~3 GB uncompressed
import os

RAW_DIR  = f'{BASE}/data/raw'
TARBALL  = f'{RAW_DIR}/GSE176078.tar.gz'
DATA_DIR = f'{RAW_DIR}/Wu_etal_2021_BRCA_scRNASeq'

if not os.path.exists(f'{DATA_DIR}/count_matrix_sparse.mtx'):
    GEO_URL = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE176nnn/GSE176078/suppl/GSE176078_Wu_etal_2021_BRCA_scRNASeq.tar.gz'
    print('Downloading dataset (~1 GB)...')
    !wget -q --show-progress -O {TARBALL} "{GEO_URL}"
    print('Extracting...')
    !mkdir -p {DATA_DIR} && tar -xzf {TARBALL} -C {DATA_DIR} --strip-components=1
    !rm {TARBALL}
    print('Dataset ready.')
else:
    print('Dataset already present.')

!ls {DATA_DIR}

---
## Stage 1 — Standard scRNA-seq Pipeline

In [ ]:
# ── Cell 6: Phase A2 — QC & Preprocessing ─────────────────────
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

sc.settings.verbosity = 1
sc.settings.figdir = f'{BASE}/results/figures/'

print('Loading raw count matrix...')
adata = sc.read_mtx(f'{DATA_DIR}/count_matrix_sparse.mtx').T
adata.obs_names = pd.read_csv(f'{DATA_DIR}/count_matrix_barcodes.tsv', header=None)[0].values
adata.var_names = pd.read_csv(f'{DATA_DIR}/count_matrix_genes.tsv',    header=None)[0].values
adata.obs       = pd.read_csv(f'{DATA_DIR}/metadata.csv', index_col=0)
print(f'Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes')

# HVG selection BEFORE normalization (seurat_v3 requires raw integer counts)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3', span=1.0)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

out = f'{BASE}/data/processed/brca_A2_preprocessed.h5ad'
adata.write_h5ad(out)
print(f'Saved: {out}  |  HVGs: {adata.var["highly_variable"].sum()}')

In [ ]:
# ── Cell 7: Phase A3 — Clustering (PCA → Harmony → UMAP → Leiden) ─
import harmonypy as hm

print('Scaling HVGs and running PCA...')
adata_hvg = adata[:, adata.var['highly_variable']].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=50, svd_solver='arpack')
adata.obsm['X_pca'] = adata_hvg.obsm['X_pca']
adata.uns['pca']    = adata_hvg.uns['pca']

print('Harmony batch correction (26 patients)...')
ho = hm.run_harmony(adata.obsm['X_pca'], adata.obs,
                    vars_use=['orig.ident'], max_iter_harmony=20, random_state=42)
adata.obsm['X_pca_harmony'] = ho.Z_corr  # harmonypy>=2.0: already (n_cells, n_pcs)

print('Building neighbors + UMAP + Leiden...')
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony', random_state=42)
sc.tl.umap(adata, random_state=42)
sc.tl.leiden(adata, resolution=0.3, key_added='leiden_0.3', random_state=42)

n_clusters = adata.obs['leiden_0.3'].nunique()
print(f'Leiden clusters (res=0.3): {n_clusters}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc.pl.umap(adata, color='celltype_major', legend_loc='on data',
           legend_fontsize=7, ax=axes[0], show=False, title='Cell Type')
sc.pl.umap(adata, color='leiden_0.3', legend_loc='on data',
           ax=axes[1], show=False, title=f'Leiden ({n_clusters} clusters)')
sc.pl.umap(adata, color='subtype', ax=axes[2], show=False, title='Subtype')
plt.tight_layout()
plt.savefig(f'{BASE}/results/figures/A3_UMAP.png', dpi=120)
plt.show()

adata.write_h5ad(f'{BASE}/data/processed/brca_A3_clustered.h5ad')
print('Saved: brca_A3_clustered.h5ad')

In [ ]:
# ── Cell 8: Phase A4 — Extract Cancer Epithelial cells ────────
print('Cell type composition:')
print(adata.obs['celltype_major'].value_counts().to_string())

cancer_epi = adata[adata.obs['celltype_major'] == 'Cancer Epithelial'].copy()
print(f'\nCancer Epithelial cells: {cancer_epi.n_obs:,}')
print(f'Subtypes: {cancer_epi.obs["subtype"].value_counts().to_dict()}')
print(f'Minor types: {cancer_epi.obs["celltype_minor"].value_counts().to_dict()}')

cancer_epi.write_h5ad(f'{BASE}/data/processed/brca_A4_cancer_epi.h5ad')
print('Saved: brca_A4_cancer_epi.h5ad')

In [ ]:
# ── Cell 9: Phase A5 — Stemness Scoring + CSC DE Markers ──────
# Three complementary stemness signatures scored on each Cancer Epithelial cell.
# Top 25% = CSC-high, bottom 25% = CSC-low → Wilcoxon DE test.

adata_epi = sc.read_h5ad(f'{BASE}/data/processed/brca_A4_cancer_epi.h5ad')

SIG_PLURIPOTENCY = ['SOX2','POU5F1','NANOG','KLF4','MYC','LIN28A','LIN28B',
                    'SALL4','PRDM14','DNMT3B','UTF1','DPPA4','DPPA2',
                    'TDGF1','ZFP42','LEFTY1','LEFTY2']
SIG_EMT         = ['VIM','FN1','CDH2','TWIST1','TWIST2','SNAI1','SNAI2',
                   'ZEB1','ZEB2','FOXC2','PDGFRB','ITGB3','HMGA2',
                   'L1CAM','CD44','ALDH1A1','ALDH1A3']
SIG_BRCA_CSC    = ['CD44','PROM1','ITGA6','EPCAM','ALDH1A1','ALDH1A3',
                   'BMI1','EZH2','NOTCH1','NOTCH2','WNT5A','LGR5',
                   'AXIN2','FZD7','YAP1','TEAD1','FOXM1','SOX9','KLF5','ID1','ID3']

def score(adata, genes, name):
    avail = [g for g in genes if g in adata.var_names]
    sc.tl.score_genes(adata, avail, score_name=name, random_state=42)
    print(f'  {name}: {len(avail)}/{len(genes)} genes')

print('Scoring stemness signatures...')
score(adata_epi, SIG_PLURIPOTENCY, 'score_pluripotency')
score(adata_epi, SIG_EMT,          'score_emt')
score(adata_epi, SIG_BRCA_CSC,     'score_brca_csc')

for col in ['score_pluripotency','score_emt','score_brca_csc']:
    mu, sd = adata_epi.obs[col].mean(), adata_epi.obs[col].std()
    adata_epi.obs[f'{col}_z'] = (adata_epi.obs[col] - mu) / (sd + 1e-9)
adata_epi.obs['stemness_composite'] = (
    adata_epi.obs['score_pluripotency_z'] +
    adata_epi.obs['score_emt_z'] +
    adata_epi.obs['score_brca_csc_z']
) / 3.0

q75 = adata_epi.obs['stemness_composite'].quantile(0.75)
q25 = adata_epi.obs['stemness_composite'].quantile(0.25)
adata_epi.obs['csc_label'] = 'middle'
adata_epi.obs.loc[adata_epi.obs['stemness_composite'] >= q75, 'csc_label'] = 'csc_high'
adata_epi.obs.loc[adata_epi.obs['stemness_composite'] <= q25, 'csc_label'] = 'csc_low'
print(f'\nCSC labels: {adata_epi.obs["csc_label"].value_counts().to_dict()}')

# Differential expression: CSC-high vs CSC-low
print('\nRunning Wilcoxon DE (CSC-high vs CSC-low)...')
adata_lab = adata_epi[adata_epi.obs['csc_label'] != 'middle'].copy()
sc.tl.rank_genes_groups(adata_lab, groupby='csc_label',
                        groups=['csc_high'], reference='csc_low',
                        method='wilcoxon', n_genes=adata_lab.n_vars, key_added='de_csc')

de_df = sc.get.rank_genes_groups_df(adata_lab, group='csc_high', key='de_csc')
de_df.columns = ['gene_symbol','wilcoxon_score','pval','pval_adj','log2fc']
stage1_markers = de_df[(de_df['pval_adj'] < 0.05) & (de_df['log2fc'] > 0.5)].copy()
print(f'Stage 1 CSC markers: {len(stage1_markers)}')
print('\nTop 20:')
print(stage1_markers.head(20)[['gene_symbol','log2fc','pval_adj']].to_string(index=False))

stage1_markers.to_csv(f'{BASE}/results/tables/A5_csc_markers_DE.csv', index=False)
adata_epi.write_h5ad(f'{BASE}/data/processed/brca_A5_csc_scored.h5ad')
print('\nSaved Stage 1 markers + scored object.')

In [ ]:
# ── Cell 10: Stage 1 summary plot ─────────────────────────────
import seaborn as sns

# Recompute subset UMAP for Cancer Epithelial cells
adata_hvg2 = adata_epi[:, adata_epi.var['highly_variable']].copy()
sc.pp.scale(adata_hvg2, max_value=10)
sc.tl.pca(adata_hvg2, n_comps=30, svd_solver='arpack')
adata_epi.obsm['X_pca_subset'] = adata_hvg2.obsm['X_pca']
sc.pp.neighbors(adata_epi, n_neighbors=15, n_pcs=30, use_rep='X_pca_subset', random_state=42)
sc.tl.umap(adata_epi, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sc.pl.umap(adata_epi, color='stemness_composite', color_map='RdYlBu_r',
           title='Composite Stemness Score', ax=axes[0], show=False)
sc.pl.umap(adata_epi, color='csc_label',
           palette={'csc_high':'#d62728','middle':'#aec7e8','csc_low':'#1f77b4'},
           title='CSC Label', ax=axes[1], show=False)
sc.pl.umap(adata_epi, color='celltype_minor', legend_loc='right margin',
           legend_fontsize=8, title='Minor Cell Type', ax=axes[2], show=False)
plt.suptitle('Stage 1 — Cancer Epithelial cells', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{BASE}/results/figures/A5_stemness_UMAP.png', dpi=120, bbox_inches='tight')
plt.show()

# Volcano plot
fig, ax = plt.subplots(figsize=(8, 6))
sig = de_df['pval_adj'] < 0.05
up  = de_df['log2fc'] > 0.5
ax.scatter(de_df.loc[~sig,'log2fc'], -np.log10(de_df.loc[~sig,'pval_adj']+1e-300),
           s=3, alpha=0.3, color='grey')
ax.scatter(de_df.loc[sig&up,'log2fc'], -np.log10(de_df.loc[sig&up,'pval_adj']+1e-300),
           s=5, alpha=0.7, color='#d62728', label=f'CSC-high ({(sig&up).sum()})')
ax.scatter(de_df.loc[sig&~up,'log2fc'], -np.log10(de_df.loc[sig&~up,'pval_adj']+1e-300),
           s=5, alpha=0.7, color='#1f77b4', label=f'CSC-low ({(sig&~up).sum()})')
for _, r in stage1_markers.head(12).iterrows():
    ax.annotate(r['gene_symbol'], (r['log2fc'], -np.log10(r['pval_adj']+1e-300)), fontsize=7)
ax.axvline(0.5, color='grey', linestyle='--', lw=0.8)
ax.set_xlabel('log2FC (CSC-high / CSC-low)'); ax.set_ylabel('-log10 adj p-value')
ax.set_title('Stage 1 — CSC Differential Expression')
ax.legend()
plt.tight_layout()
plt.savefig(f'{BASE}/results/figures/A5_volcano.png', dpi=120)
plt.show()
print(f'Stage 1 complete. {len(stage1_markers)} CSC markers identified.')

---
## Stage 2 — Geneformer Transformer Analysis

In [ ]:
# ── Cell 11: Phase G1+G2 — Tokenize CSC-labelled cells ────────
# Geneformer requirements:
#   - Raw integer counts (all genes, no feature selection)
#   - var['ensembl_id'] — gene symbols (tokenizer maps internally)
#   - obs['n_counts'] — total UMI per cell
#   - max sequence length 512 tokens (fits GPU; captures top 512 expressed genes)

import collections
from geneformer import TranscriptomeTokenizer
from datasets import load_from_disk

# Reload scored object and get labels
adata_epi = sc.read_h5ad(f'{BASE}/data/processed/brca_A5_csc_scored.h5ad')
labelled_barcodes = adata_epi.obs[adata_epi.obs['csc_label'] != 'middle'].index
label_df = adata_epi.obs.loc[labelled_barcodes, ['csc_label','subtype','celltype_minor']]
print(f'Labelled cells: {len(labelled_barcodes):,}  '
      f"({(label_df['csc_label']=='csc_high').sum()} high, "
      f"{(label_df['csc_label']=='csc_low').sum()} low)")

# Load raw counts (all 29,733 genes)
print('Loading raw counts...')
raw = sc.read_mtx(f'{DATA_DIR}/count_matrix_sparse.mtx').T
raw.obs_names = pd.read_csv(f'{DATA_DIR}/count_matrix_barcodes.tsv', header=None)[0].values
raw.var_names = pd.read_csv(f'{DATA_DIR}/count_matrix_genes.tsv',    header=None)[0].values

raw_lab = raw[labelled_barcodes, :].copy()
raw_lab.var['ensembl_id'] = raw_lab.var_names.values
raw_lab.obs['n_counts']   = np.asarray(raw_lab.X.sum(axis=1)).flatten().astype(int)
raw_lab.obs = raw_lab.obs.join(label_df, how='left')
raw_lab = raw_lab[raw_lab.obs['n_counts'] > 0].copy()

h5ad_path = f'{BASE}/data/geneformer/input/brca_csc_labelled.h5ad'
raw_lab.write_h5ad(h5ad_path)
print(f'Saved: {raw_lab.n_obs:,} cells × {raw_lab.n_vars:,} genes')

# Tokenize — max 512 tokens (top 512 expressed genes per cell)
print('Tokenizing (max 512 tokens per cell)...')
tk = TranscriptomeTokenizer(
    custom_attr_name_dict={'csc_label':'csc_label','subtype':'subtype','celltype_minor':'celltype_minor'},
    nproc=2,
    model_input_size=512,
    model_version='V2',
)
tk.tokenize_data(
    data_directory=f'{BASE}/data/geneformer/input/',
    output_directory=f'{BASE}/data/geneformer/',
    output_prefix='brca_csc',
    file_format='h5ad',
)

dataset = load_from_disk(f'{BASE}/data/geneformer/brca_csc.dataset')
print(f'Tokenized: {len(dataset):,} cells')
print(f'Columns: {dataset.column_names}')
print(f'Avg token length: {sum(len(d["input_ids"]) for d in dataset)/len(dataset):.0f}')
print(f'Label distribution: {dict(collections.Counter(dataset["csc_label"]))}')

In [ ]:
# ── Cell 12: Phase G3 — Fine-tune Geneformer on GPU ───────────
import pickle, torch
from geneformer import Classifier

MODEL_DIR = f'{BASE}/geneformer_repo/Geneformer-V2-104M_CLcancer'
OUT_DIR   = f'{BASE}/results/geneformer_finetuned'
PREFIX    = 'brca_csc'
DATASET   = f'{BASE}/data/geneformer/brca_csc.dataset'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cpu':
    print('WARNING: GPU not available. Training will be slow (~4–8 hrs). Enable GPU runtime.')

cc = Classifier(
    classifier='cell',
    cell_state_dict={'state_key': 'csc_label', 'states': 'all'},
    training_args={
        'num_train_epochs':           5,
        'learning_rate':              5e-5,
        'lr_scheduler_type':          'cosine',
        'warmup_ratio':               0.05,
        'per_device_train_batch_size': 12,
        'per_device_eval_batch_size':  12,
        'weight_decay':               0.01,
        'output_dir':                 OUT_DIR,
        'evaluation_strategy':        'epoch',
        'save_strategy':              'epoch',
        'load_best_model_at_end':     True,
        'metric_for_best_model':      'eval_macro_f1',
        'logging_steps':              50,
        'fp16':                       (device == 'cuda'),
    },
    freeze_layers=6,
    num_crossval_splits=1,
    split_sizes={'train': 0.8, 'valid': 0.1, 'test': 0.1},
    forward_batch_size=100,
    nproc=2,
    ngpu=1 if device == 'cuda' else 0,
    model_version='V2',
)

print('Preparing data splits...')
cc.prepare_data(input_data_file=DATASET, output_directory=OUT_DIR, output_prefix=PREFIX)

from datasets import load_from_disk as lfd
train_data = lfd(f'{OUT_DIR}/{PREFIX}_labeled_train.dataset')
test_data  = lfd(f'{OUT_DIR}/{PREFIX}_labeled_test.dataset')
id_class_path = f'{OUT_DIR}/{PREFIX}_id_class_dict.pkl'
with open(id_class_path, 'rb') as f:
    id_class_dict = pickle.load(f)
print(f'Train: {len(train_data):,}  Test: {len(test_data):,}')
print(f'Label mapping: {id_class_dict}')

print('\nFine-tuning Geneformer-V2-104M_CLcancer...')
metrics = cc.train_classifier(
    model_directory=MODEL_DIR,
    num_classes=len(id_class_dict),
    train_data=train_data,
    eval_data=test_data,
    output_directory=OUT_DIR,
    predict=True,
)
print(f'\nTraining metrics: {metrics}')

In [ ]:
# ── Cell 13: Phase G4 — Extract Attention Weights ─────────────
# Forward-pass CSC-high cells through fine-tuned model with output_attentions=True.
# Average attention across all heads and layers → per-gene importance score.

import pickle
import torch
from transformers import BertForSequenceClassification
from datasets import load_from_disk

print('Loading fine-tuned model...')
model = BertForSequenceClassification.from_pretrained(OUT_DIR)
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'Model on: {device}')

# Load token → Ensembl ID mapping
with open(f'{BASE}/geneformer_repo/geneformer/token_dictionary_gc104M.pkl', 'rb') as f:
    token_dict = pickle.load(f)
id_to_ensembl = {v: k for k, v in token_dict.items()}

# Load Ensembl → gene symbol mapping
with open(f'{BASE}/geneformer_repo/geneformer/ensembl_mapping_dict_gc104M.pkl', 'rb') as f:
    symbol_to_ensembl = pickle.load(f)
ensembl_to_symbol = {v: k for k, v in symbol_to_ensembl.items()}

# Filter to CSC-high cells
dataset = load_from_disk(f'{BASE}/data/geneformer/brca_csc.dataset')
csc_high_ds = dataset.filter(lambda x: x['csc_label'] == 'csc_high')
print(f'CSC-high cells for attention extraction: {len(csc_high_ds):,}')

# Accumulate attention per gene across cells (use up to 500 cells for speed)
gene_attn_sum   = {}
gene_attn_count = {}
N_CELLS = min(500, len(csc_high_ds))
BATCH   = 8 if device == 'cuda' else 4

print(f'Extracting attention from {N_CELLS} CSC-high cells (batch={BATCH})...')
for i in range(0, N_CELLS, BATCH):
    batch    = csc_high_ds[i:i+BATCH]
    # Pad sequences to same length within batch
    max_len  = max(len(ids) for ids in batch['input_ids'])
    padded   = [ids + [0]*(max_len - len(ids)) for ids in batch['input_ids']]
    input_ids = torch.tensor(padded, dtype=torch.long).to(device)
    attn_mask = (input_ids != 0).long()

    with torch.no_grad():
        out = model(input_ids=input_ids, attention_mask=attn_mask, output_attentions=True)

    # Stack all layers: (n_layers, batch, heads, seq, seq)
    stacked      = torch.stack(out.attentions, dim=0)
    # Mean over layers + heads → (batch, seq, seq)
    mean_attn    = stacked.mean(dim=(0, 2))
    # Column sum: how much attention each token RECEIVES → (batch, seq)
    gene_importance = mean_attn.sum(dim=1)

    for b in range(gene_importance.shape[0]):
        tokens = input_ids[b].cpu().tolist()
        scores = gene_importance[b].cpu().tolist()
        for tok, sc in zip(tokens, scores):
            if tok in id_to_ensembl and tok != 0:
                ens = id_to_ensembl[tok]
                gene_attn_sum[ens]   = gene_attn_sum.get(ens, 0) + sc
                gene_attn_count[ens] = gene_attn_count.get(ens, 0) + 1

    if (i // BATCH) % 10 == 0:
        print(f'  Processed {min(i+BATCH, N_CELLS)}/{N_CELLS} cells')

# Average and rank
gene_avg = {g: gene_attn_sum[g]/gene_attn_count[g] for g in gene_attn_sum}
attn_df  = (pd.DataFrame.from_dict(gene_avg, orient='index', columns=['attention_score'])
              .sort_values('attention_score', ascending=False)
              .reset_index().rename(columns={'index':'ensembl_id'}))
attn_df['gene_symbol'] = attn_df['ensembl_id'].map(ensembl_to_symbol)

attn_df.to_csv(f'{BASE}/results/tables/G4_geneformer_gene_ranking.csv', index=False)
print(f'\nTop 20 genes by Geneformer attention:')
print(attn_df.dropna(subset=['gene_symbol']).head(20)[['gene_symbol','attention_score']].to_string(index=False))

In [ ]:
# ── Cell 14: Phase G5 — Compare Stage 1 vs Stage 2 ────────────
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

# Load both marker sets
stage1_df = pd.read_csv(f'{BASE}/results/tables/A5_csc_markers_DE.csv')
stage1_genes = set(stage1_df['gene_symbol'].dropna().head(200))
stage2_genes = set(attn_df.dropna(subset=['gene_symbol'])['gene_symbol'].head(200))

shared       = stage1_genes & stage2_genes
stage2_only  = stage2_genes - stage1_genes
stage1_only  = stage1_genes - stage2_genes
jaccard      = len(shared) / len(stage1_genes | stage2_genes)

print('='*55)
print('STAGE 1 vs STAGE 2 — CSC MARKER COMPARISON (top 200)')
print('='*55)
print(f'Stage 1 (DE)               : {len(stage1_genes)} genes')
print(f'Stage 2 (Geneformer attn)  : {len(stage2_genes)} genes')
print(f'Shared by both methods     : {len(shared)} genes')
print(f'Stage 2 only (novel)       : {len(stage2_only)} genes')
print(f'Stage 1 only               : {len(stage1_only)} genes')
print(f'Jaccard similarity         : {jaccard:.3f}')

print(f'\nTop 20 NOVEL candidates (Stage 2 only — not found by DE):')
novel_ranked = (attn_df[attn_df['gene_symbol'].isin(stage2_only)]
                .dropna(subset=['gene_symbol']).head(20))
print(novel_ranked[['gene_symbol','attention_score']].to_string(index=False))

# Save tables
pd.DataFrame(sorted(shared),      columns=['gene']).to_csv(f'{BASE}/results/tables/G5_shared_markers.csv',      index=False)
pd.DataFrame(sorted(stage2_only), columns=['gene']).to_csv(f'{BASE}/results/tables/G5_novel_stage2_markers.csv', index=False)
pd.DataFrame(sorted(stage1_only), columns=['gene']).to_csv(f'{BASE}/results/tables/G5_stage1_only_markers.csv',  index=False)

# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Venn diagram
try:
    from matplotlib_venn import venn2
    venn2([stage1_genes, stage2_genes], set_labels=('Stage 1\n(DE)', 'Stage 2\n(Geneformer)'), ax=axes[0])
except ImportError:
    categories = ['Stage 1 only\n(DE)', 'Shared\n(both)', 'Stage 2 only\n(Attention)']
    counts = [len(stage1_only), len(shared), len(stage2_only)]
    axes[0].bar(categories, counts, color=['#FF6B6B','#4ECDC4','#45B7D1'])
    for i, c in enumerate(counts):
        axes[0].text(i, c+1, str(c), ha='center', fontweight='bold')
axes[0].set_title(f'Marker overlap (Jaccard={jaccard:.3f})')

# Top 20 Stage 2 markers
top20 = attn_df.dropna(subset=['gene_symbol']).head(20)
axes[1].barh(top20['gene_symbol'][::-1], top20['attention_score'][::-1], color='steelblue')
axes[1].set_xlabel('Mean Attention Score')
axes[1].set_title('Top 20 genes — Geneformer attention (Stage 2)')

# Top 20 Stage 1 markers
top20_s1 = stage1_df.head(20)
axes[2].barh(top20_s1['gene_symbol'][::-1], top20_s1['log2fc'][::-1], color='#d62728')
axes[2].set_xlabel('log2 Fold Change')
axes[2].set_title('Top 20 genes — Differential Expression (Stage 1)')

plt.suptitle('CSC Gene Program: Stage 1 (DE) vs Stage 2 (Geneformer)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{BASE}/results/figures/G5_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nAnalysis complete. All results saved.')

In [ ]:
# ── Cell 15: Download results to local machine ─────────────────
# Zip all result files and download them
import shutil
shutil.make_archive('/content/CSC_results', 'zip', f'{BASE}/results')

from google.colab import files
files.download('/content/CSC_results.zip')
print('Results zip downloaded.')
print(f'\nContents of {BASE}/results/')
for root, dirs, filenames in os.walk(f'{BASE}/results'):
    for fn in filenames:
        path = os.path.join(root, fn)
        size = os.path.getsize(path)
        rel  = path.replace(BASE+'/', '')
        print(f'  {rel}  ({size/1024:.0f} KB)')

---
## Results Summary

| File | Contents |
|---|---|
| `results/tables/A5_csc_markers_DE.csv` | Stage 1: DE-based CSC markers with log2FC + adj p-value |
| `results/tables/G4_geneformer_gene_ranking.csv` | Stage 2: All genes ranked by Geneformer attention score |
| `results/tables/G5_shared_markers.csv` | Genes confirmed by **both** methods |
| `results/tables/G5_novel_stage2_markers.csv` | **Novel** CSC candidates found only by Geneformer |
| `results/figures/G5_comparison.png` | Venn diagram + ranked bar charts |
| `results/figures/A5_volcano.png` | Stage 1 volcano plot |
| `results/figures/A3_UMAP.png` | Cell type UMAP (all 100k cells) |

### Interpreting the novel Stage 2 markers
Genes in `G5_novel_stage2_markers.csv` are statistically enriched in CSC-high cells (that's why the classifier learned them) but were not among the top 200 differential expression hits. These are the most interesting candidates — they likely reflect **co-expression context** and **gene regulatory network** relationships that DE tests miss. Prioritise them for:
- Literature validation (PubMed + GeneCards)
- Surface protein check (Human Protein Atlas — surface proteome)
- Cross-cancer validation in Sub-project B